# Data Cleaning
Fixes all data quality issues and standardizes location fields across every dataset.
Outputs cleaned files to `data/processed/`.

**Location standardization:**
- County-level: `geoid` (5-digit FIPS), `state_fips` (2-digit), `county_fips` (3-digit), `state_abbr`, `county_name`
- Point-level: `lat`, `lon`, `state_abbr`

In [ ]:
import pandas as pd
import re
from pathlib import Path

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
OUT  = DATA / 'processed'
OUT.mkdir(exist_ok=True)

# State name <-> abbreviation lookup
STATE_ABBR = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC', 'Puerto Rico': 'PR',
}
ABBR_STATE = {v: k for k, v in STATE_ABBR.items()}

def to_geoid(state_fips, county_fips):
    """Zero-pad and combine state+county FIPS into 5-digit geoid string."""
    return str(int(state_fips)).zfill(2) + str(int(county_fips)).zfill(3)

def parse_wiki_location(loc_str):
    """Extract decimal lat/lon from Wikipedia unicode location string."""
    # Format: '40°55′45″N 79°27′59″W / 40.92917°N 79.46639°W'
    match = re.search(r'([\d.]+)°N\s+([\d.]+)°W', loc_str)
    if match:
        return float(match.group(1)), -float(match.group(2))
    return None, None

print('Setup complete.')

---
## 1. nina_demand_data.csv
**Fixes:** Drop 15 ghost rows (empty City rows where `state_id` is null), drop 3 empty columns
**Note:** County rows legitimately have `city_id` = null — they use `county_id`
**Standardize:** `latitude`/`longitude` → `lat`/`lon`; split City and County into separate files

In [ ]:
demand = pd.read_csv(ROOT / 'nina_demand_data.csv', low_memory=False)
print(f'Raw: {demand.shape}')

# Drop true ghost rows: state_id is null (15 completely empty City rows)
# Note: County rows legitimately have null city_id — they use county_id instead
demand = demand.dropna(subset=['state_id'])

# Drop empty columns
demand = demand.drop(columns=['com_buildings', 'com_area_sqft', 'ind_establishments'])

# Standardize: rename coords
demand = demand.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

county_cols = [c for c in demand.columns if c.startswith('county_')]

demand_cities = (
    demand[demand['level'] == 'City']
    .drop(columns=county_cols)
    .reset_index(drop=True)
)
demand_counties = (
    demand[demand['level'] == 'County']
    .reset_index(drop=True)
)

demand_cities.to_csv(OUT / 'demand_cities.csv', index=False)
demand_counties.to_csv(OUT / 'demand_counties.csv', index=False)
print(f'demand_cities: {demand_cities.shape}')
print(f'demand_counties: {demand_counties.shape}')

---
## 2. arcgis_decommissioned_plants.csv
**Fixes:** Rename typo column, convert Unix ms timestamp
**Standardize:** `Plant_latitude`/`Plant_longitude` → `lat`/`lon`; `Plant_State` → `state_abbr`

In [ ]:
plants = pd.read_csv(DATA / 'decommissioned coal or gas power plants/arcgis_decommissioned_plants.csv')

plants = plants.rename(columns={
    'Fuel_Catagory':  'fuel_category',
    'Plant_latitude':  'lat',
    'Plant_longitude': 'lon',
    'Plant_State':     'state_abbr',
    'Plant_Name':      'plant_name',
    'Plant_ID':        'plant_id',
    'Retirement_Year': 'retirement_year',
    'Retirement_Month':'retirement_month',
    'Retirement_Date': 'retirement_date',
    'Entity_Name':     'entity_name',
    'Entity_ID':       'entity_id',
    'Sector':          'sector',
    'Generator_ID':    'generator_id',
    'Nameplate_Capacity__MW_': 'capacity_mw',
    'Technology':      'technology',
    'Energy_Source':   'energy_source',
    'Prime_Mover':     'prime_mover',
    'Data_Source':     'data_source',
    'ObjectId':        'object_id',
})
plants['retirement_date'] = pd.to_datetime(plants['retirement_date'], unit='ms')

plants.to_csv(OUT / 'decommissioned_plants.csv', index=False)
print(f'decommissioned_plants: {plants.shape}')
plants.head(2)

---
## 3. wikipedia decommissioned coal plant tables
**Fixes:** Parse retirement date, drop Ref column, numeric capacity
**Standardize:** Extract decimal lat/lon from unicode Location field; full state name → `state_abbr`; combine tables

In [ ]:
wiki2 = pd.read_csv(DATA / 'decommissioned coal or gas power plants/wikipedia_decommissioned_coal_plants_table_2.csv')
wiki4 = pd.read_csv(DATA / 'decommissioned coal or gas power plants/wikipedia_decommissioned_coal_plants_table_4.csv')
wiki4['table_source'] = 'table_4'
wiki2['table_source'] = 'table_2'
wiki = pd.concat([wiki2, wiki4], ignore_index=True)

# Parse location
coords = wiki['Location'].apply(lambda x: pd.Series(parse_wiki_location(str(x)), index=['lat', 'lon']))
wiki = pd.concat([wiki, coords], axis=1)

# Standardize state
wiki['state_abbr'] = wiki['State'].map(STATE_ABBR)

# Fix dates and types
wiki['retired_date'] = pd.to_datetime(wiki['Retired'], errors='coerce')
wiki['capacity_mw']  = pd.to_numeric(wiki['Capacity(MW)'], errors='coerce')

# Drop noisy/redundant cols
wiki = wiki.drop(columns=['Location', 'State', 'Capacity(MW)', 'Retired', 'Ref'], errors='ignore')

# Rename remaining cols to snake_case
wiki = wiki.rename(columns={
    'Name': 'plant_name',
    'Majority Owner': 'majority_owner',
    'Fuel type': 'fuel_type',
    'Capacity Factor': 'capacity_factor',
    'Annual Generation (GWh)': 'annual_gen_gwh',
    'CO2 emissions (Tons/year) [2][3]': 'co2_emissions_tons_yr',
    'CO2 emissions/ Annual Generation': 'co2_per_gen',
})

wiki.to_csv(OUT / 'wiki_decommissioned_coal.csv', index=False)
print(f'wiki_decommissioned_coal: {wiki.shape}')
print(f'Lat/lon parsed: {wiki["lat"].notna().sum()} of {len(wiki)}')
wiki.head(2)

---
## 4. osm_data_centers_usa.csv
**Fixes:** No fixable issues (OSM sparsity is inherent)
**Standardize:** `addr_state` → `state_abbr`; column renames

In [ ]:
dc = pd.read_csv(DATA / 'datacenter_locations/osm_data_centers_usa.csv')

dc = dc.rename(columns={
    'addr_city':  'city',
    'addr_state': 'state_abbr',
    'operator':   'operator',
})
# Replace empty strings with NaN
dc = dc.replace('', pd.NA)

dc.to_csv(OUT / 'data_centers.csv', index=False)
print(f'data_centers: {dc.shape}')
print(f'Have state_abbr: {dc["state_abbr"].notna().sum()} of {len(dc)}')

---
## 5. fema_nfip_claims_by_county.csv
**Fixes:** Separate state-level aggregate rows from county rows
**Standardize:** `countyCode` → `geoid`; derive `state_fips`, `county_fips`; rename `state` → `state_abbr`

In [ ]:
fema_claims = pd.read_csv(DATA / 'fema_flood_zones/fema_nfip_claims_by_county.csv',
                           dtype={'countyCode': str})

fema_claims = fema_claims.rename(columns={
    'countyCode':            'geoid',
    'state':                 'state_abbr',
    'total_claims':          'total_claims',
    'total_building_damage': 'total_building_damage_$',
    'total_content_damage':  'total_content_damage_$',
})

# State-level aggregates (no county assigned)
fema_claims_state  = fema_claims[fema_claims['geoid'].isna()].reset_index(drop=True)
fema_claims_county = fema_claims[fema_claims['geoid'].notna()].copy()

# Zero-pad geoid and derive state/county fips
fema_claims_county['geoid']       = fema_claims_county['geoid'].str.zfill(5)
fema_claims_county['state_fips']  = fema_claims_county['geoid'].str[:2]
fema_claims_county['county_fips'] = fema_claims_county['geoid'].str[2:]

fema_claims_county.to_csv(OUT / 'fema_claims_county.csv', index=False)
fema_claims_state.to_csv(OUT / 'fema_claims_state.csv', index=False)
print(f'fema_claims_county: {fema_claims_county.shape}')
print(f'fema_claims_state (aggregates): {fema_claims_state.shape}')

---
## 6. fema_nfip_community_status_book.csv
**Fixes:** CRS columns only apply to enrolled communities — no fix needed, document it
**Standardize:** `county` (ALL CAPS) → `county_name` (title case); `state` → `state_abbr`

In [ ]:
fema_comm = pd.read_csv(DATA / 'fema_flood_zones/fema_nfip_community_status_book.csv')

fema_comm = fema_comm.rename(columns={
    'county': 'county_name',
    'state':  'state_abbr',
})
fema_comm['county_name'] = fema_comm['county_name'].str.title()

fema_comm.to_csv(OUT / 'fema_community_status.csv', index=False)
print(f'fema_community_status: {fema_comm.shape}')
print(f'CRS-enrolled: {fema_comm["classRating"].notna().sum()} communities')

---
## 7. Transmission lines
**Fixes:** Strip BOM from us_transmission_lines column names
**Standardize:** Rename columns to snake_case; use `us_transmission_lines.csv` as primary (fuller)
Note: `coordinate_us_transmission_lines.csv` is identical in both folders — use one copy

In [ ]:
# Fuller attribute file — encoding='utf-8-sig' strips BOM automatically
trans = pd.read_csv(DATA / 'transmission_line/us_transmission_lines.csv', encoding='utf-8-sig')
trans = trans.rename(columns={
    'Object ID (Internal)': 'object_id_internal',
    'Object ID': 'object_id',
    'Transmission Line ID': 'line_id',
    'Line Type': 'line_type',
    'Operational Status': 'status',
    'North American Industry Classification System (NAICS) Code': 'naics_code',
    'North American Industry Classification System (NAICS) Description': 'naics_desc',
    'Data Source': 'data_source',
    'Source Date': 'source_date',
    'Validation Method': 'validation_method',
    'Validation Date': 'validation_date',
    'Transmission Line Owner': 'owner',
    'Voltage (Kilovolts)': 'voltage_kv',
    'Voltage Class': 'voltage_class',
    'Inferred Attributes': 'inferred',
    'Substation 1': 'substation_1',
    'Substation 2': 'substation_2',
    'Shape Length': 'shape_length',
})

# Geometry-enriched coordinate file
trans_coords = pd.read_csv(DATA / 'transmission_line/coordinate_us_transmission_lines.csv')
trans_coords = trans_coords.rename(columns={
    'OBJECTID': 'object_id', 'ID': 'line_id', 'TYPE': 'line_type',
    'STATUS': 'status', 'OWNER': 'owner', 'VOLTAGE': 'voltage_kv',
    'VOLT_CLASS': 'voltage_class', 'INFERRED': 'inferred',
    'SUB_1': 'substation_1', 'SUB_2': 'substation_2',
    'NAICS_DESC': 'naics_desc', 'SHAPE__Len': 'shape_length',
})
# Standardize remaining coordinate columns to lowercase
trans_coords.columns = trans_coords.columns.str.lower()

trans.to_csv(OUT / 'transmission_lines.csv', index=False)
trans_coords.to_csv(OUT / 'transmission_lines_coords.csv', index=False)
print(f'transmission_lines: {trans.shape}')
print(f'transmission_lines_coords: {trans_coords.shape}')

---
## 8. eia860_2024_nuclear_plant_locations.csv
**Fixes:** Drop 8 fully empty columns
**Standardize:** `Latitude`/`Longitude` → `lat`/`lon`; `State` → `state_abbr`; all column names to snake_case

In [ ]:
nuclear = pd.read_csv(DATA / 'nrc_reactor_locations/eia860_2024_nuclear_plant_locations.csv')

# Drop fully empty columns
empty_cols = [c for c in nuclear.columns if nuclear[c].isna().all()]
nuclear = nuclear.drop(columns=empty_cols)

# Standardize key location columns
nuclear = nuclear.rename(columns={
    'Latitude':  'lat',
    'Longitude': 'lon',
    'State':     'state_abbr',
    'City':      'city',
    'County':    'county_name',
})
# snake_case all remaining columns
nuclear.columns = (
    nuclear.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

nuclear.to_csv(OUT / 'nuclear_plants.csv', index=False)
print(f'nuclear_plants: {nuclear.shape}')
print(f'Dropped empty cols: {empty_cols}')

---
## 9. osm_wetlands_usa.csv
**Fixes:** 
- Dropped rows with duplicate data, except possibly for id
- Dropped 'natural' column (all values are 'wetland')
- Normalized 'wetland' column (converting null values to 'unclassified', manually converting type names, catching spelling mistakes)
- Removed wetlands data outside of the U.S. (using county boundaries data)
**Columns:** `id`,`lat`,`lon`,`name`,`wetland_type`,`geometry`,`geo_id`,`county_name`,`state_fip_code`,`county_fip_code`

See wetlands_eda.ipynb in 'notebooks' folder

---
## 10. usgs_seismic_hazard_by_county.csv
**Fixes:** Drop `risk_coeff_pga` (100% empty)
**Standardize:** Already clean — `geoid`, `state_fips`, `county_fips`, `state_abbr`, `centroid_lat`, `centroid_lon`
Rename centroid coords to `lat`/`lon` for consistency

In [ ]:
seismic = pd.read_csv(DATA / 'usgs_seismic_hazard/usgs_seismic_hazard_by_county.csv',
                       dtype={'geoid': str, 'state_fips': str, 'county_fips': str})

seismic = seismic.drop(columns=['risk_coeff_pga'])
seismic = seismic.rename(columns={'centroid_lat': 'lat', 'centroid_lon': 'lon'})

# Ensure zero-padded FIPS
seismic['geoid']       = seismic['geoid'].str.zfill(5)
seismic['state_fips']  = seismic['state_fips'].str.zfill(2)
seismic['county_fips'] = seismic['county_fips'].str.zfill(3)

seismic.to_csv(OUT / 'seismic_hazard.csv', index=False)
print(f'seismic_hazard: {seismic.shape}')

---
## 11. Census income + housing (2022 ACS)
**Fixes:** None
**Standardize:** Add `geoid`; ensure zero-padded FIPS; rename `county_name` to drop state suffix

In [ ]:
census = pd.read_csv(DATA / 'census_population_county_boundaries/census_2022_acs_county_income_housing.csv',
                      dtype={'state_fips': str, 'county_fips': str})

census['state_fips']  = census['state_fips'].str.zfill(2)
census['county_fips'] = census['county_fips'].str.zfill(3)
census['geoid']       = census['state_fips'] + census['county_fips']

# county_name is 'Autauga County, Alabama' — split off state name
census[['county_name', 'state_name']] = census['county_name'].str.rsplit(', ', n=1, expand=True)
census['state_abbr'] = census['state_name'].map(STATE_ABBR)

census.to_csv(OUT / 'census_income_housing.csv', index=False)
print(f'census_income_housing: {census.shape}')
census.head(2)

---
## 12. population_2024.csv
**Fixes:** None
**Standardize:** `state`/`county` (int, unpadded) → `state_fips`/`county_fips` (zero-padded str); add `geoid`; add `state_abbr`

In [ ]:
pop2024 = pd.read_csv(DATA / 'population/population_2024.csv')

pop2024['state_fips']  = pop2024['state'].astype(str).str.zfill(2)
pop2024['county_fips'] = pop2024['county'].astype(str).str.zfill(3)
pop2024['geoid']       = pop2024['state_fips'] + pop2024['county_fips']
pop2024['state_abbr']  = pop2024['state_name'].map(STATE_ABBR)
pop2024 = pop2024.drop(columns=['state', 'county'])
pop2024 = pop2024.rename(columns={'estimate_population': 'population_2024'})

pop2024.to_csv(OUT / 'population_2024.csv', index=False)
print(f'population_2024: {pop2024.shape}')
pop2024.head(2)

---
## 13. population_2018_2024.csv
**Fixes:** None
**Standardize:** `STATE`/`COUNTY` → `state_fips`/`county_fips` (zero-padded); add `geoid`; snake_case columns

In [ ]:
pop_ts = pd.read_csv(DATA / 'population/population_2018_2024.csv',
                      dtype={'STATE': str, 'COUNTY': str})

pop_ts['state_fips']  = pop_ts['STATE'].str.zfill(2)
pop_ts['county_fips'] = pop_ts['COUNTY'].str.zfill(3)
pop_ts['geoid']       = pop_ts['state_fips'] + pop_ts['county_fips']
pop_ts['state_abbr']  = pop_ts['STNAME'].map(STATE_ABBR)
pop_ts = pop_ts.drop(columns=['STATE', 'COUNTY'])
pop_ts.columns = pop_ts.columns.str.lower()

pop_ts.to_csv(OUT / 'population_timeseries.csv', index=False)
print(f'population_timeseries: {pop_ts.shape}')

---
## 14. county_land_use_proxy_census.csv
**Fixes:** None
**Standardize:** Already has `geoid`, `state_fips`, `county_fips` — just ensure zero-padding

In [ ]:
land_use = pd.read_csv(DATA / 'nlcd_land_cover/county_land_use_proxy_census.csv',
                        dtype={'geoid': str, 'state_fips': str, 'county_fips': str})

land_use['geoid']       = land_use['geoid'].str.zfill(5)
land_use['state_fips']  = land_use['state_fips'].str.zfill(2)
land_use['county_fips'] = land_use['county_fips'].str.zfill(3)

land_use.to_csv(OUT / 'land_use.csv', index=False)
print(f'land_use: {land_use.shape}')

---
## 15. padus4_protected_areas_national.csv
**Fixes:** None (5 missing unit names — negligible)
**Standardize:** `ST_Name` → `state_abbr`; snake_case columns

In [ ]:
protected = pd.read_csv(DATA / 'protected_areas_database/padus4_protected_areas_national.csv')

protected = protected.rename(columns={
    'OBJECTID':    'object_id',
    'Unit_Nm':     'unit_name',
    'Pub_Access':  'public_access',
    'GAP_Sts':     'gap_status',
    'IUCN_Cat':    'iucn_category',
    'MngTp_Desc':  'manager_type',
    'MngNm_Desc':  'manager_name',
    'GIS_Acres':   'gis_acres',
    'ST_Name':     'state_abbr',
    'Mang_Type':   'manager_type_code',
    'Des_Tp_Desc': 'designation_type',
})

protected.to_csv(OUT / 'protected_areas.csv', index=False)
print(f'protected_areas: {protected.shape}')

---
## Summary of output files

In [ ]:
print('Files written to data/processed/:')
for f in sorted(OUT.glob('*.csv')):
    df = pd.read_csv(f, nrows=0)
    print(f'  {f.name}: {len(df.columns)} cols')